In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG = "clinical_trial_supply_chain"

BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

SOURCE_TABLE = f"{BRONZE}.bronze_inventory_snapshot"
TARGET_TABLE = f"{SILVER}.fact_inventory"

In [0]:
inventory_df = (
    spark.table(SOURCE_TABLE)
    .select(
        "snapshot_date",
        "site_id",
        "study_id",
        "drug_type",
        "qty_on_hand",
        "qty_reserved",
        "qty_expired",
        "inventory_adjustment"
    )
)

In [0]:
inventory_df = (
    inventory_df
    .withColumn(
        "available_stock",
        F.col("qty_on_hand")
        - F.col("qty_reserved")
        - F.col("qty_expired")
    )
)

In [0]:
(
    inventory_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE)
)

In [0]:
display(
    spark.table(TARGET_TABLE)
)